# Train and calibrate URTI cough models
Attach the private prepared dataset and enable a GPU. This notebook compares several experimental cough classifiers, selects by validation ROC-AUC, calibrates on validation data only, and reports held-out test metrics. It is not a clinical diagnostic system.

In [ ]:
import importlib.util, subprocess, sys, shutil
from pathlib import Path
packages = {'numpy': 'numpy', 'pandas': 'pandas', 'librosa': 'librosa>=0.10,<0.12',
            'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib'}
missing = [pkg for mod, pkg in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
if importlib.util.find_spec('tensorflow') is None:
    raise RuntimeError('Use a Kaggle TensorFlow-compatible GPU environment; TensorFlow is missing.')
if shutil.which('ffmpeg') is None:
    raise RuntimeError('FFmpeg is missing. Install FFmpeg in this Kaggle runtime before continuing.')
import tensorflow as tf
print('TensorFlow:', tf.__version__, 'GPUs:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'Enable a GPU accelerator first.'


In [ ]:
# Set an explicit path here only if auto-discovery finds multiple datasets.
DATA_ROOT = None
if DATA_ROOT is None:
    candidates = [p.parent for p in Path('/kaggle/input').rglob('manifest.csv')
                  if (p.parent / 'preprocessing.json').exists() and (p.parent / 'train.py').exists()]
    assert len(candidates) == 1, f'Expected one prepared dataset, found: {candidates}'
    DATA_ROOT = candidates[0]
DATA_ROOT = Path(DATA_ROOT)
import pandas as pd
manifest = pd.read_csv(DATA_ROOT / 'manifest.csv')
display(manifest.groupby(['split', 'label']).size().unstack(fill_value=0))
assert manifest.uuid.is_unique
assert manifest.groupby('sha256')['split'].nunique().max() == 1
print('Dataset:', DATA_ROOT)


## Train, compare, calibrate, and evaluate
Preprocessing runs on CPU first. Training uses validation ROC-AUC to choose a checkpoint. Platt and isotonic calibration are fitted on the validation split only; the held-out test split is used once for final metrics. If preprocessing fails, inspect `outputs/preprocessing_failures.json` before proceeding.

In [ ]:
subprocess.run([sys.executable, '-u', str(DATA_ROOT / 'train.py'),
                '--data', str(DATA_ROOT), '--output', '/kaggle/working/outputs',
                '--epochs', '60'], check=True)


In [ ]:
import json
from IPython.display import Image, display, FileLink
print(json.dumps(json.loads(Path('/kaggle/working/outputs/metrics.json').read_text()), indent=2))
display(Image(filename='/kaggle/working/outputs/evaluation.png'))
display(FileLink('/kaggle/working/urti_model_bundle.zip'))
print('Also available in the notebook output files: urti_model_bundle.zip')
